In [0]:
#Image metadata and maintenance logs, and finishing Bronze
from pyspark.sql.functions import current_timestamp, lit

CATALOG = "capstone_supply_chain"

image_records = [
    {"image_id": "IMG-QI-001", "inspection_id": "QI-2026-001", "part_number": "TIR-ALL-2233",
     "image_type": "defect_photo", "camera_id": "CAM-DOCK5-01", "resolution": "4032x3024",
     "file_size_kb": 3842, "format": "JPEG", "capture_timestamp": "2026-03-12T14:22:15Z",
     "description": "Sidewall bulge visible at 180 degree position on tire unit #TIR-997",
     "defect_class": "sidewall_deformation", "confidence_score": 0.94,
     "file_path": f"/Volumes/{CATALOG}/bronze/landing_images/QI-001-sidewall.jpg"},
    {"image_id": "IMG-QI-002", "inspection_id": "QI-2026-002", "part_number": "SET-LTH-9900",
     "image_type": "color_comparison", "camera_id": "CAM-LAB-SPECTRO", "resolution": "2048x2048",
     "file_size_kb": 5120, "format": "TIFF", "capture_timestamp": "2026-03-20T09:45:30Z",
     "description": "Color comparison: left=master sample (OK), right=lot LC-2026-088 (FAIL Delta E 3.8)",
     "defect_class": "color_mismatch", "confidence_score": 0.98,
     "file_path": f"/Volumes/{CATALOG}/bronze/landing_images/QI-002-color.tiff"},
    {"image_id": "IMG-QI-003", "inspection_id": "QI-2026-003", "part_number": "INJ-GDI-5566",
     "image_type": "xray_scan", "camera_id": "XRAY-NDT-01", "resolution": "1024x1024",
     "file_size_kb": 8192, "format": "DICOM", "capture_timestamp": "2026-05-10T11:30:00Z",
     "description": "X-ray of injector nozzle showing orifice diameter deviation +0.015mm",
     "defect_class": "dimensional_deviation", "confidence_score": 0.87,
     "file_path": f"/Volumes/{CATALOG}/bronze/landing_images/QI-003-nozzle-xray.dcm"},
    {"image_id": "IMG-QI-004", "inspection_id": "QI-2026-004", "part_number": "VLV-ENG-3345",
     "image_type": "microscopy", "camera_id": "MICRO-MET-01", "resolution": "2560x1920",
     "file_size_kb": 6400, "format": "PNG", "capture_timestamp": "2026-04-15T16:00:00Z",
     "description": "Metallurgical cross-section at 200x magnification. Microstructure normal.",
     "defect_class": "none", "confidence_score": 1.0,
     "file_path": f"/Volumes/{CATALOG}/bronze/landing_images/QI-004-metallurgy.png"},
    {"image_id": "IMG-QI-005", "inspection_id": "QI-2026-005", "part_number": "TRB-CHR-6678",
     "image_type": "thermal_image", "camera_id": "FLIR-T530", "resolution": "640x480",
     "file_size_kb": 2048, "format": "JPEG", "capture_timestamp": "2026-05-18T13:15:45Z",
     "description": "Thermal image at 150K RPM showing 12C hotspot on bearing housing",
     "defect_class": "thermal_anomaly", "confidence_score": 0.91,
     "file_path": f"/Volumes/{CATALOG}/bronze/landing_images/QI-005-thermal.jpg"},
]

df_images = (spark.createDataFrame(image_records)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_file_format", lit("image_metadata"))
)

df_images.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.raw_inspection_images")
print(f"✅ raw_inspection_images ({len(image_records)} records)")

In [0]:
#Maintenance logs (cell 2)
import os
from pyspark.sql.functions import current_timestamp, lit

CATALOG = "capstone_supply_chain"
unstruct_vol = f"/Volumes/{CATALOG}/bronze/landing_unstructured"

log_records = []
for f in os.listdir(unstruct_vol):
    filepath = os.path.join(unstruct_vol, f)
    if not os.path.isfile(filepath):      # skip any sub-folders
        continue
    with open(filepath, "r") as fh:
        content = fh.read()
    log_records.append({
        "filename": f,
        "file_path": filepath,
        "raw_content": content,
        "content_length": len(content),
        "document_type": "maintenance_log",
    })

df_logs = (spark.createDataFrame(log_records)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_file_format", lit("unstructured_text"))
)

df_logs.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.raw_maintenance_logs")
print(f"✅ raw_maintenance_logs ({len(log_records)} records)")

In [0]:
#Bronze layer summary (cell 3)
CATALOG = "capstone_supply_chain"

print("BRONZE LAYER - All tables:")
for t in spark.sql(f"SHOW TABLES IN {CATALOG}.bronze").collect():
    count = spark.table(f"{CATALOG}.bronze.{t.tableName}").count()
    print(f"  {t.tableName}: {count} rows")